# SU Framework Training — HCMUT Dataset

This notebook trains and evaluates the SU (Supervised & Unsupervised) Framework based on the paper *Predicting running time of aerodynamic jobs in HPC system by combining supervised and unsupervised learning method* (s42774-021-00077-8).

It implements clustering of jobs based on User and Job name strings, searches for KNN, and uses a local SVR with an adjustment factor $\alpha$ to minimize underestimation.

In [6]:
import os
import sys
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
from SU import prepare_data_SU, SUFramework

# Optional: Set seed for reproducibility
# np.random.seed(42)

In [7]:
df = pd.read_csv(r'../output_csv/HCMUT-SuperNodeXP-2017-1.0.swf.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15886 entries, 0 to 15885
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   job_id                    15886 non-null  float64
 1   submit_time               15886 non-null  float64
 2   wait_time                 15886 non-null  float64
 3   run_time                  15886 non-null  float64
 4   num_allocated_processors  15886 non-null  float64
 5   avg_cpu_time_used         15886 non-null  float64
 6   used_memory               15886 non-null  float64
 7   requested_processors      15886 non-null  float64
 8   requested_time            15886 non-null  float64
 9   requested_memory          15886 non-null  float64
 10  status                    15886 non-null  float64
 11  user_id                   15886 non-null  float64
 12  group_id                  15886 non-null  float64
 13  executable_id             15886 non-null  float64
 14  queue_

In [8]:
# Prepare data natively for SU Framework (without upfront scaling)
train_df, val_df, test_df = prepare_data_SU(df, statuss=1)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

Train size: 2749, Val size: 344, Test size: 344


In [9]:
# Define SU hyperparameters based on the paper's tuning process
K_NEIGHBORS = 20           # Number of similar jobs to train the local SVR
ALPHA = 1.15               # Adjustment factor to reduce underestimation
NUM_CLUSTERS_PER_USER = 3  # K-means++ / Agglomerative clusters per user

# Initialize Framework
su_model = SUFramework(k=K_NEIGHBORS, alpha=ALPHA, num_clusters_per_user=NUM_CLUSTERS_PER_USER)

# 1. Unsupervised Phase: Calculate distances, cluster historical jobs
print("Starting Training (Clustering + Local feature setup)...")
su_model.fit(train_df)
print("Training setup complete.")

Starting Training (Clustering + Local feature setup)...


Training setup complete.


In [ ]:
# 2. Supervised Phase: KNN lookup and local SVR training per test sample
print("Evaluating on Test Set...")
metrics = su_model.evaluate(test_df)

print(f"\n{'='*50}")
print(f"SU Framework Test Results — HCMUT Dataset")
print(f"{'='*50}")
print(f"MAE:                  {metrics['MAE']:.4f}")
print(f"RMSE:                 {metrics['RMSE']:.4f}")
print(f"R² Score:             {metrics['R² Score']:.4f}")
print(f"Underestimation Rate: {metrics['Underestimation Rate']:.4%}")
print(f"Average Pred Accuracy:{metrics['APA']:.4f}")
print(f"Inference Time:       {metrics['Inference Time']:.6f} s/sample")
print(f"{'='*50}")

# Save single-run result as this method doesn't need 100 iterations of random initializations
s = pd.DataFrame([metrics])
results_df.to_csv('../output_HCMUT/SU_Framework_results.csv', index=False)

Evaluating on Test Set...



SU Framework Test Results — HCMUT Dataset
MAE:                  66264.8330
RMSE:                 136306.3255
R² Score:             -0.3489
Underestimation Rate: 27.3256%
Average Pred Accuracy:0.5785
Inference Time:       0.046349 s/sample
